# 🎙️ VoxCPM2 — 내 목소리로 내레이션 만들기 (mystery-unlock-style)

본인 목소리를 [VoxCPM2](https://huggingface.co/openbmb/VoxCPM2)(코드·가중치 Apache-2.0)로 복제해, 에피소드 대본을 한 줄씩 읽힌 WAV 를 만듭니다.

**준비**: 메뉴 `런타임 → 런타임 유형 변경 → T4 GPU` 를 고른 뒤 위에서부터 차례로 실행하세요.

| 단계 | 언제 | 결과 |
|---|---|---|
| **1단계 내 목소리 등록** | 처음 한 번 | `my_voice_ref.wav` (Drive 에 보관) |
| **2단계 에피소드 생성** | 에피소드마다 | `<에피소드>_voice.zip` (s01_c01.wav …) |

- 생성한 zip 은 PC 에서 `python scripts/voice.py import episodes/<ep> <zip>` 로 가져옵니다.
- ⚠️ **본인 목소리만** 씁니다(`config/voice.json → consent: own`). 다른 사람·크리에이터 목소리 복제는 하지 않습니다.
- 한국어 품질은 먼저 [VoxCPM 데모](https://huggingface.co/spaces/OpenBMB/VoxCPM-Demo)에서 짧게 들어보고 시작하는 걸 권합니다.

In [ ]:
# GPU · 파이썬 버전 확인 (VoxCPM2: Python 3.10~3.12, CUDA 12+)
import sys
print(sys.version)
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "⚠️ GPU 없음: 런타임 → 런타임 유형 변경 → T4 GPU"

In [ ]:
!pip install -q voxcpm soundfile

In [ ]:
# 저장 위치 — Drive 를 쓰면 연결이 끊겨도 결과가 남고, 다음에 이어서 생성합니다
USE_DRIVE = True
import os, re, json, shutil, time, zipfile
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    WORK = '/content/drive/MyDrive/mystery-unlock-style-voice'
else:
    WORK = '/content/mystery-unlock-style-voice'
os.makedirs(f'{WORK}/voice', exist_ok=True)
print('저장 위치:', WORK)

In [ ]:
# 모델 불러오기 (처음 한 번 수 GB 다운로드)
from voxcpm import VoxCPM
import soundfile as sf
from IPython.display import Audio, display
from google.colab import files
model = VoxCPM.from_pretrained("openbmb/VoxCPM2", load_denoiser=False)
SR = model.tts_model.sample_rate
print('샘플레이트', SR)

## 1단계 — 내 목소리 등록 (처음 한 번)

1. 조용한 방에서 아래 `REF_TEXT` 문장을 **그대로** 읽어 10~30초 녹음합니다(휴대폰 녹음 앱도 됩니다. wav/m4a/mp3).
   이 문장은 저장소의 `config/voice.json → reference_text` 와 글자 하나까지 같아야 합니다.
2. 아래 셀을 실행해 녹음 파일을 올립니다. Drive 에 `voice/my_voice_ref.wav` 로 저장됩니다.
3. 그 다음 셀로 시험 문장을 들어보고, 마음에 들지 않으면 다시 녹음해 올립니다(잡음·울림이 적을수록 좋습니다).

In [ ]:
REF_TEXT = "1948년 12월 1일 아침, 해변에서 한 남자가 발견됩니다. 그의 이름이 밝혀지기까지는 무려 73년이 걸렸습니다."
REF = f"{WORK}/voice/my_voice_ref.wav"
print("녹음 파일을 올리세요 (wav/m4a/mp3)")
up = files.upload()
name, data = next(iter(up.items()))
raw = f"/content/{name}"
open(raw, "wb").write(data)
# 모노 24 kHz WAV 로 변환, 앞뒤 무음 제거
!ffmpeg -loglevel error -y -i "{raw}" -af "silenceremove=start_periods=1:start_threshold=-50dB,areverse,silenceremove=start_periods=1:start_threshold=-50dB,areverse" -ac 1 -ar 24000 "{REF}"
open(f"{WORK}/voice/my_voice_ref.txt", "w", encoding="utf-8").write(REF_TEXT)
display(Audio(REF))

In [ ]:
# 시험 문장 — 내 목소리처럼 들리는지 확인
TEST = "그런데 이 남자에겐 이상한 점이 있었습니다. 지갑도, 신분증도, 모자도 없었죠."
wav = model.generate(text=TEST, prompt_wav_path=REF, prompt_text=REF_TEXT, reference_wav_path=REF,
                     cfg_value=2.0, inference_timesteps=10, seed=42)
sf.write("/content/test.wav", wav, SR)
display(Audio("/content/test.wav"))

## 2단계 — 에피소드 내레이션 생성 (에피소드마다)

1. PC 에서 `python scripts/voice.py pack episodes/<ep>` → `episodes/<ep>/voice/pack.json`
2. 아래 셀에서 pack.json 을 올리고 생성합니다. **이미 만든 줄은 건너뜁니다**(문장이 같으면 다시 만들지 않음).
   대본을 고쳐 다시 pack 하면 바뀐 줄만 새로 만듭니다.
3. 특정 줄 발음이 이상하면 대본에서 `자막 || 읽을 문장` 으로 읽는 법을 적고 다시 pack → 생성.
   또는 Drive 의 `projects/<ep>/cache/` 에서 그 줄의 wav 를 지우고 `SEED` 를 바꿔 다시 실행합니다.

In [ ]:
PACK_IN_DRIVE = ""   # 예: /content/drive/MyDrive/mystery-unlock-style-voice/pack.json (비우면 업로드)
if PACK_IN_DRIVE:
    pack = json.load(open(PACK_IN_DRIVE, encoding="utf-8"))
else:
    print("pack.json 을 올리세요")
    up = files.upload()
    pack = json.loads(next(iter(up.values())).decode("utf-8"))
PROJECT, V = pack["project"], pack["voice"]
assert V.get("consent") == "own", "consent 가 own 이 아닙니다 — 본인 목소리만 사용합니다"
REF = f"{WORK}/voice/my_voice_ref.wav"
assert os.path.exists(REF), "1단계(내 목소리 등록)를 먼저 하세요"
REF_TEXT = open(f"{WORK}/voice/my_voice_ref.txt", encoding="utf-8").read().strip()
if REF_TEXT != V.get("reference_text", "").strip():
    print("⚠️ 녹음 문장과 config/voice.json 의 reference_text 가 다릅니다. 둘을 같게 맞추세요.")
CACHE = f"{WORK}/projects/{PROJECT}/cache"
os.makedirs(CACHE, exist_ok=True)
print(PROJECT, len(pack["chunks"]), "줄,", V["mode"])

In [ ]:
# 생성 (끊기면 이 셀만 다시 실행 → 남은 줄부터 이어서)
SEED = V.get("seed", 42)
t0 = time.time()
todo = [c for c in pack["chunks"] if not os.path.exists(f"{CACHE}/{c['key']}.wav")]
print(f"남은 줄 {len(todo)}/{len(pack['chunks'])}")
for i, c in enumerate(todo, 1):
    kw = dict(text=c["text"], reference_wav_path=REF, cfg_value=V.get("cfg_value", 2.0),
              inference_timesteps=V.get("inference_timesteps", 10), seed=SEED)
    if V.get("mode") == "ultimate_clone":
        kw.update(prompt_wav_path=REF, prompt_text=REF_TEXT, text=re.sub(r"^\([^)]*\)", "", c["text"]))
    wav = model.generate(**kw)
    tmp = f"{CACHE}/{c['key']}.part.wav"
    sf.write(tmp, wav, SR)
    os.replace(tmp, f"{CACHE}/{c['key']}.wav")
    print(f"  {i}/{len(todo)} {c['id']} {c['caption']}  ({time.time() - t0:.0f}s)")
print("완료")

In [ ]:
# 몇 줄 들어보기
for c in pack["chunks"][:3]:
    print(c["id"], c["caption"]); display(Audio(f"{CACHE}/{c['key']}.wav"))

In [ ]:
# zip 으로 묶기 (s01_c01.wav … 이름으로) → Drive 저장, 필요하면 다운로드
zp = f"{WORK}/{PROJECT}_voice.zip"
with zipfile.ZipFile(zp, "w") as z:
    for c in pack["chunks"]:
        z.write(f"{CACHE}/{c['key']}.wav", f"{c['id']}.wav")
print(zp, os.path.getsize(zp) // 1_000_000, "MB")
print(f"PC 에서: python scripts/voice.py import episodes/{PROJECT} <다운로드한 zip>")
# files.download(zp)